#### Notebook 12 — mlflow_tracking

##### 1. Purpose

This notebook will independently:

``` text

Load persisted train/validation/test splits
              ↓
Build tokenizer + pretrained Transformer
              ↓
Fine-tune Transformer
              ↓
Evaluate final model
              ↓
Package raw-text inference model
              ↓
Log experiment to MLflow
              ↓
Log parameters + metrics + artifacts + model
              ↓
Reload logged model
              ↓
Verify raw-text inference

```

This notebook will not depend on Notebook 11 having been run.

That is important for our production-style design.


##### 2. What should MLflow track?

For traditional ML, we tracked things like:

``` text

model type
hyperparameters
accuracy
F1
model artifact

```

For our Transformer, reproducibility requires more information:

``` text

Experiment
│
├── Model configuration
│   ├── pretrained model
│   ├── max sequence length
│   ├── dropout
│   └── number of classes
│
├── Training configuration
│   ├── learning rate
│   ├── batch size
│   ├── weight decay
│   ├── max epochs
│   ├── patience
│   └── random seed
│
├── Dataset information
│   ├── train rows
│   ├── validation rows
│   └── test rows
│
├── Metrics
│   ├── validation loss
│   ├── validation accuracy
│   ├── test loss
│   ├── test accuracy
│   ├── macro F1
│   └── weighted F1
│
├── Diagnostic artifacts
│   ├── training_history.csv
│   ├── classification_report.json
│   ├── confusion_matrix.csv
│   └── test_predictions.csv
│
└── Serving-ready model
    ├── fine-tuned Transformer
    ├── tokenizer
    ├── classifier
    └── label mapping

```

That last section is especially important.

We don't want future endpoint to require callers to manually tokenize text.

We eventually want:

``` text

"Account locked after many attempts"
                 ↓
          registered model
                 ↓
Login + confidence + probabilities

```

##### 3. Production architecture

``` text

support_ticket_nlp/
│
├── 11_transformer_ticket_classifier
├── 12_mlflow_tracking
├── 13_model_registration
├── 14_model_deployment
├── 15_endpoint_testing
│
└── src/
    ├── project_config.py
    ├── data_preparation.py
    ├── feature_engineering.py
    ├── model_training.py
    ├── model_evaluation.py
    ├── neural_network.py
    ├── transformer_classifier.py
    └── transformer_inference.py      ← NEW

```

Why separate them?

transformer_classifier.py → training architecture and training logic

transformer_inference.py → production inference contract

This will become valuable in Notebooks 13–15.

##### 4. imports

In [0]:
import json
import os
import tempfile

import mlflow
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from mlflow.models import infer_signature

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

from src.project_config import (
    RANDOM_SEED,
    CLASS_NAMES,
    TEXT_COL,
    TARGET_COL,

    TRANSFORMER_MODEL_NAME,
    TRANSFORMER_MAX_LENGTH,
    TRANSFORMER_BATCH_SIZE,
    TRANSFORMER_LEARNING_RATE,
    TRANSFORMER_WEIGHT_DECAY,
    TRANSFORMER_MAX_EPOCHS,
    TRANSFORMER_EARLY_STOPPING_PATIENCE,
    TRANSFORMER_DROPOUT,

    MLFLOW_EXPERIMENT_NAME,
    MLFLOW_RUN_NAME,
    MLFLOW_MODEL_ARTIFACT_PATH,
)

from src.data_preparation import (
    load_modeling_dataset,
    split_modeling_dataset,
)

from src.transformer_classifier import (
    set_transformer_seed,
    create_transformer_model,
    create_transformer_dataloader,
    count_parameters,
    fine_tune_transformer,
    evaluate_transformer,
)

from src.transformer_inference import (
    TransformerTicketPyFunc,
)

##### 5. Environment verification

In [0]:
#Because MLflow APIs can differ between Databricks Runtime versions, I like explicitly recording versions:

print(
    "MLflow version:",
    mlflow.__version__,
)

print(
    "PyTorch version:",
    torch.__version__,
)

In [0]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Device:",
    device,
)

##### 6. Set experiment

In [0]:
mlflow.set_experiment(
    MLFLOW_EXPERIMENT_NAME
)

In [0]:
experiment = (
    mlflow.get_experiment_by_name(
        MLFLOW_EXPERIMENT_NAME
    )
)

print(
    "Experiment ID:",
    experiment.experiment_id,
)

print(
    "Experiment name:",
    experiment.name,
)

##### 7. Load persisted splits

In [0]:
modeling_df = load_modeling_dataset(spark)

train_df, validation_df, test_df = (
    split_modeling_dataset(
        modeling_df
    )
)

train_pd = train_df.toPandas()
validation_pd = validation_df.toPandas()
test_pd = test_df.toPandas()

print(
    "Train:",
    len(train_pd)
)

print(
    "Validation:",
    len(validation_pd)
)

print(
    "Test:",
    len(test_pd)
)

##### 8. Prepare labels

In [0]:
label_to_id = {
    label: index
    for index, label
    in enumerate(CLASS_NAMES)
}

id_to_label = {
    index: label
    for label, index
    in label_to_id.items()
}

y_train = (
    train_pd[TARGET_COL]
    .map(label_to_id)
    .to_numpy()
)

y_validation = (
    validation_pd[TARGET_COL]
    .map(label_to_id)
    .to_numpy()
)

y_test = (
    test_pd[TARGET_COL]
    .map(label_to_id)
    .to_numpy()
)

assert not pd.isna(y_train).any()
assert not pd.isna(y_validation).any()
assert not pd.isna(y_test).any()

#####  9. Create model and DataLoaders

In [0]:
set_transformer_seed(
    RANDOM_SEED
)

tokenizer, model = (
    create_transformer_model(
        num_classes=len(
            CLASS_NAMES
        )
    )
)

model = model.to(
    device
)

In [0]:
train_loader = (
    create_transformer_dataloader(
        texts=train_pd[TEXT_COL],
        labels=y_train,
        tokenizer=tokenizer,
        batch_size=(
            TRANSFORMER_BATCH_SIZE
        ),
        shuffle=True,
    )
)

validation_loader = (
    create_transformer_dataloader(
        texts=validation_pd[TEXT_COL],
        labels=y_validation,
        tokenizer=tokenizer,
        batch_size=(
            TRANSFORMER_BATCH_SIZE
        ),
        shuffle=False,
    )
)

test_loader = (
    create_transformer_dataloader(
        texts=test_pd[TEXT_COL],
        labels=y_test,
        tokenizer=tokenizer,
        batch_size=(
            TRANSFORMER_BATCH_SIZE
        ),
        shuffle=False,
    )
)

##### 10. Parameter information

In [0]:
parameter_info = count_parameters(model)

parameter_info

##### 11. Start MLflow run and fine-tune

In [0]:
# ---------------------------------------------------------
# Start one MLflow run
# ---------------------------------------------------------

if mlflow.active_run() is not None:
    mlflow.end_run()

with mlflow.start_run(
    run_name=MLFLOW_RUN_NAME
) as run:

    run_id = run.info.run_id

    print(
        "MLflow Run ID:",
        run_id,
    )

    # -----------------------------------------------------
    # Log parameters
    # -----------------------------------------------------

    mlflow.log_params(
        {
            "model_name":
                TRANSFORMER_MODEL_NAME,

            "max_length":
                TRANSFORMER_MAX_LENGTH,

            "batch_size":
                TRANSFORMER_BATCH_SIZE,

            "learning_rate":
                TRANSFORMER_LEARNING_RATE,

            "weight_decay":
                TRANSFORMER_WEIGHT_DECAY,

            "max_epochs":
                TRANSFORMER_MAX_EPOCHS,

            "early_stopping_patience":
                TRANSFORMER_EARLY_STOPPING_PATIENCE,

            "dropout":
                TRANSFORMER_DROPOUT,

            "random_seed":
                RANDOM_SEED,

            "num_classes":
                len(CLASS_NAMES),

            "train_rows":
                len(train_pd),

            "validation_rows":
                len(validation_pd),

            "test_rows":
                len(test_pd),

            "total_parameters":
                parameter_info[
                    "total_parameters"
                ],

            "trainable_parameters":
                parameter_info[
                    "trainable_parameters"
                ],

            "frozen_parameters":
                parameter_info[
                    "frozen_parameters"
                ],
        }
    )

    # -----------------------------------------------------
    # Fine-tune model
    # -----------------------------------------------------

    model, training_history = (
        fine_tune_transformer(
            model=model,
            train_loader=train_loader,
            validation_loader=(
                validation_loader
            ),
            device=device,
            max_epochs=(
                TRANSFORMER_MAX_EPOCHS
            ),
            patience=(
                TRANSFORMER_EARLY_STOPPING_PATIENCE
            ),
        )
    )

    # -----------------------------------------------------
    # Log epoch-level metrics
    # -----------------------------------------------------

    history_df = pd.DataFrame(
        training_history
    )

    for _, row in history_df.iterrows():

        epoch = int(
            row["epoch"]
        )

        mlflow.log_metrics(
            {
                "train_loss":
                    float(
                        row["train_loss"]
                    ),

                "train_accuracy":
                    float(
                        row["train_accuracy"]
                    ),

                "validation_loss":
                    float(
                        row[
                            "validation_loss"
                        ]
                    ),

                "validation_accuracy":
                    float(
                        row[
                            "validation_accuracy"
                        ]
                    ),
            },
            step=epoch,
        )

    # -----------------------------------------------------
    # Final evaluation
    # -----------------------------------------------------

    criterion = nn.CrossEntropyLoss()

    validation_result = (
        evaluate_transformer(
            model=model,
            dataloader=validation_loader,
            criterion=criterion,
            device=device,
        )
    )

    test_result = (
        evaluate_transformer(
            model=model,
            dataloader=test_loader,
            criterion=criterion,
            device=device,
        )
    )

    test_labels = (
        test_result["labels"]
    )

    test_predictions = (
        test_result["predictions"]
    )

    test_probabilities = (
        test_result["probabilities"]
    )

    test_accuracy = accuracy_score(
        test_labels,
        test_predictions,
    )

    test_macro_f1 = f1_score(
        test_labels,
        test_predictions,
        average="macro",
    )

    test_weighted_f1 = f1_score(
        test_labels,
        test_predictions,
        average="weighted",
    )

    # -----------------------------------------------------
    # Log final metrics
    # -----------------------------------------------------

    mlflow.log_metrics(
        {
            "final_validation_loss":
                float(
                    validation_result[
                        "loss"
                    ]
                ),

            "final_validation_accuracy":
                float(
                    validation_result[
                        "accuracy"
                    ]
                ),

            "test_loss":
                float(
                    test_result[
                        "loss"
                    ]
                ),

            "test_accuracy":
                float(
                    test_accuracy
                ),

            "test_macro_f1":
                float(
                    test_macro_f1
                ),

            "test_weighted_f1":
                float(
                    test_weighted_f1
                ),

            "epochs_trained":
                float(
                    len(history_df)
                ),
        }
    )

    # -----------------------------------------------------
    # Build evaluation artifacts
    # -----------------------------------------------------

    report = classification_report(
        test_labels,
        test_predictions,
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )

    confusion = confusion_matrix(
        test_labels,
        test_predictions,
    )

    confusion_df = pd.DataFrame(
        confusion,
        index=CLASS_NAMES,
        columns=CLASS_NAMES,
    )

    prediction_df = (
        test_pd[
            [
                "ticket_id",
                TEXT_COL,
                TARGET_COL,
            ]
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    prediction_df[
        "predicted_category"
    ] = [
        id_to_label[index]
        for index
        in test_predictions
    ]

    prediction_df[
        "confidence"
    ] = (
        test_probabilities.max(
            axis=1
        )
    )

    prediction_df[
        "correct"
    ] = (
        prediction_df[
            TARGET_COL
        ]
        ==
        prediction_df[
            "predicted_category"
        ]
    )

    # -----------------------------------------------------
    # Log evaluation artifacts
    # -----------------------------------------------------

    with tempfile.TemporaryDirectory() as temp_dir:

        history_path = os.path.join(
            temp_dir,
            "training_history.csv",
        )

        confusion_path = os.path.join(
            temp_dir,
            "confusion_matrix.csv",
        )

        predictions_path = os.path.join(
            temp_dir,
            "test_predictions.csv",
        )

        report_path = os.path.join(
            temp_dir,
            "classification_report.json",
        )

        history_df.to_csv(
            history_path,
            index=False,
        )

        confusion_df.to_csv(
            confusion_path,
        )

        prediction_df.to_csv(
            predictions_path,
            index=False,
        )

        with open(
            report_path,
            "w",
            encoding="utf-8",
        ) as file:

            json.dump(
                report,
                file,
                indent=2,
            )

        mlflow.log_artifacts(
            temp_dir,
            artifact_path="evaluation",
        )

    # -----------------------------------------------------
    # Package serving model
    # -----------------------------------------------------

    with tempfile.TemporaryDirectory() as model_dir:

        transformer_dir = os.path.join(
            model_dir,
            "transformer",
        )

        tokenizer_dir = os.path.join(
            model_dir,
            "tokenizer",
        )

        classifier_path = os.path.join(
            model_dir,
            "classifier.pt",
        )

        metadata_path = os.path.join(
            model_dir,
            "metadata.json",
        )

        model.transformer.save_pretrained(
            transformer_dir
        )

        tokenizer.save_pretrained(
            tokenizer_dir
        )

        torch.save(
            model.classifier.state_dict(),
            classifier_path,
        )

        metadata = {
            "class_names":
                CLASS_NAMES,

            "max_length":
                TRANSFORMER_MAX_LENGTH,

            "hidden_size":
                model.transformer
                .config
                .hidden_size,

            "model_name":
                TRANSFORMER_MODEL_NAME,
        }

        with open(
            metadata_path,
            "w",
            encoding="utf-8",
        ) as file:

            json.dump(
                metadata,
                file,
                indent=2,
            )

        input_example = pd.DataFrame(
            {
                TEXT_COL: [
                    (
                        "Account locked "
                        "after many attempts"
                    ),
                    (
                        "Please cancel "
                        "my subscription"
                    ),
                ]
            }
        )

        output_example = pd.DataFrame(
            {
                "predicted_category": [
                    "Login",
                    "Cancellation",
                ],
                "confidence": [
                    0.8,
                    0.8,
                ],
                "probability_Billing": [
                    0.05,
                    0.05,
                ],
                "probability_Cancellation": [
                    0.05,
                    0.8,
                ],
                "probability_Login": [
                    0.85,
                    0.05,
                ],
                "probability_Technical": [
                    0.05,
                    0.10,
                ],
            }
        )

        signature = infer_signature(
            input_example,
            output_example,
        )

        model_info = (
            mlflow.pyfunc.log_model(
                name=MLFLOW_MODEL_ARTIFACT_PATH,                
                python_model= TransformerTicketPyFunc(),
                artifacts={
                    "transformer": transformer_dir,
                    "tokenizer": tokenizer_dir,
                    "classifier": classifier_path,
                    "metadata":  metadata_path,
                },

                code_paths=["src"],
                pip_requirements=[
                "mlflow==3.0.1",
                "torch==2.7.0",
                "transformers==4.51.3",
                "pandas==2.2.3",
                "numpy==2.1.3",
                "safetensors==0.6.2",
                ],
                signature= signature,
                input_example=  input_example,
            )
        )

        print(
            "Logged model info:",
            model_info
        )

        print(
            "Logged model URI:",
            model_info.model_uri
        )

        model_uri = (
            model_info.model_uri
        )

    print(
        "Completed MLflow Run ID:",
        run_id
    )

    print(
        "Model URI:",
        model_uri
    )

This is crucial.

saving the fine-tuned Transformer, not merely remembering: sentence-transformers/all-MiniLM-L6-v2

Otherwise serving would reload the original pretrained weights and lose our task-specific fine-tuning.

Before logging, we'll produce a representative output later during reload verification. For the MLflow signature, we can define the expected output schema through a small example:

##### 19. Log serving-ready PyFunc model

Conceptually, MLflow now packages:

``` text

MLflow Model
│
├── TransformerTicketPyFunc
│
├── fine-tuned MiniLM weights
│
├── tokenizer
│
├── classifier head
│
├── metadata
│   ├── classes
│   └── max length
│
└── signature

```
This is much more useful than simply logging a .pt file.

In [0]:
print(
    "Active run:",
    mlflow.active_run()
)

print(
    "Run ID:",
    run_id
)

print(
    "Model URI:",
    model_uri
)

##### 21. Reload the model from MLflow

 After the with mlflow.start_run(...) block has finished, do something very important:Don't trust a model simply because MLflow successfully logged it.

In [0]:
##### Reload it.

loaded_model = (
    mlflow.pyfunc.load_model(
        model_uri
    )
)

In [0]:
inference_input = pd.DataFrame(
    {
        TEXT_COL: [
            "Account locked after many attempts",
            "My internet connection keeps dropping",
            "Please cancel my subscription",
        ]
    }
)

inference_output = (
    loaded_model.predict(
        inference_input
    )
)

display(
    inference_output
)

##### 22. Final verification

In [0]:
assert (
    len(inference_output)
    ==
    len(inference_input)
)

assert (
    "predicted_category"
    in inference_output.columns
)

assert (
    "confidence"
    in inference_output.columns
)

assert set(
    inference_output[
        "predicted_category"
    ]
).issubset(
    set(CLASS_NAMES)
)

assert (
    inference_output[
        "confidence"
    ]
    .between(
        0.0,
        1.0,
    )
    .all()
)

print(
    "MLflow model reload "
    "verification passed."
)

##### 23. Display final experiment summary

In [0]:
summary_df = pd.DataFrame(
    [
        {
            "run_id":
                run_id,

            "model":
                TRANSFORMER_MODEL_NAME,

            "train_rows":
                len(train_pd),

            "validation_rows":
                len(
                    validation_pd
                ),

            "test_rows":
                len(test_pd),

            "test_accuracy":
                test_accuracy,

            "test_macro_f1":
                test_macro_f1,

            "test_weighted_f1":
                test_weighted_f1,

            "model_uri":
                model_uri,
        }
    ]
)

display(
    summary_df
)

##### Key Learnings

For the final notebook, I would keep the key learnings concise.

MLflow tracks more than model accuracy. A reproducible Transformer experiment needs the pretrained architecture, hyperparameters, dataset information, metrics, training history, tokenizer, class mapping and trained model parameters.

Notebook 12 retrains independently. It does not reuse the model object created by Notebook 11. It reconstructs the persisted data splits, creates the Transformer, fine-tunes it and logs the resulting run.

The tokenizer is part of the model contract. Raw support-ticket text cannot be passed directly to BertModel; the same tokenizer behavior used by the model must be available during inference.

We save the fine-tuned Transformer weights. Saving only the original Hugging Face model name would reconstruct the pretrained model and lose the changes made by fine-tuning.

PyFunc creates a clean serving contract.

Instead of:

caller
- → tokenize
- → tensors
- → model
- → softmax
- → decode class

we expose:

caller
- → ticket_text
- → MLflow model
- → predicted category

Reload testing is part of model validation. Successful log_model() does not guarantee that a model can later be loaded and used correctly. Notebook 12 explicitly reloads the logged artifact and performs raw-text inference.


artifact structure now represents a complete production model:

``` text

MLFLOW RUN
│
├── evaluation/
│   │
│   ├── training_history.csv
│   │     └─ How did training progress?
│   │
│   ├── classification_report.json
│   │     └─ How did each class perform?
│   │
│   ├── confusion_matrix.csv
│   │     └─ Which classes were confused?
│   │
│   └── test_predictions.csv
│         └─ What happened for each ticket?
│
└── model/
    │
    ├── artifacts/
    │   │
    │   ├── tokenizer/
    │   │     └─ text → tokens/IDs
    │   │
    │   ├── transformer/
    │   │     ├─ config.json
    │   │     └─ fine-tuned weights
    │   │
    │   ├── classifier.pt
    │   │     └─ 384 → 4 classifier weights
    │   │
    │   └── metadata.json
    │         └─ classes/max length/etc.
    │
    ├── MLmodel
    │     └─ MLflow model manifest
    │
    ├── python_model.pkl
    │     └─ PyFunc inference wrapper
    │
    ├── input_example.json
    │     └─ example raw model input
    │
    ├── serving_input_example.json
    │     └─ serving request example
    │
    ├── requirements.txt
    ├── python_env.yaml
    └── conda.yaml
          └─ runtime/dependencies

```

- Training/evaluation evidence → evaluation/
- Model components            → artifacts/
- MLflow loading instructions → MLmodel + python_model.pkl
- Environment reproducibility → requirements/environment files
- Serving contract            → input/signature examples

##### Conclusion

A good final notebook conclusion is:

Notebook 12 converted the fine-tuned support-ticket Transformer from an in-memory training object into a reproducible MLflow experiment. The run records the Transformer architecture, training hyperparameters, dataset sizes, epoch-level training metrics, final validation/test metrics and diagnostic evaluation artifacts.

The fine-tuned Transformer weights, tokenizer, classification head and class metadata were packaged behind an MLflow PyFunc interface so downstream applications can provide raw ticket_text without reproducing tokenization or PyTorch inference logic.

Finally, the logged model was reloaded from its MLflow run URI and tested independently, verifying that the tracked artifact is usable for inference rather than merely successfully serialized.

##### What changes compared with Notebook 11?

This distinction is worth remembering:

``` ytext

Notebook 11
"What is Transformer fine-tuning,
 and does our implementation work?"

              ↓

Notebook 12
"Can this training result be tracked,
 reproduced, packaged and reloaded?"

              ↓

Notebook 13
"Which tracked model should become
 our governed Unity Catalog model?"

 ```